In [2]:
import json
import time
from kafka import KafkaProducer

def json_serializer(data):
    return json.dumps(data).encode('utf-8')

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers = [server],
    value_serializer = json_serializer
)

producer.send('testtopic', {'test1': 'value1'})

producer.flush()

In [44]:
from kafka import KafkaConsumer

consumer = KafkaConsumer('green_2019_10', bootstrap_servers=[server])

In [48]:
for m in consumer:
    print(m)
    break

ConsumerRecord(topic='green_2019_10', partition=0, leader_epoch=1, offset=5, timestamp=1754224207634, timestamp_type=0, key=None, value=b'{"lpep_pickup_datetime": "2019-10-01 00:26:02", "lpep_dropoff_datetime": "2019-10-01 00:39:58", "PULocationID": 112, "DOLocationID": 196, "passenger_count": 1.0, "trip_distance": 5.88, "tip_amount": 0.0}', headers=[], checksum=None, serialized_key_size=-1, serialized_value_size=203, serialized_header_size=-1)


In [28]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2019-10.csv.gz

--2025-08-03 20:20:41--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2019-10.csv.gz
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/ea580e9e-555c-4bd0-ae73-43051d8e7c0b?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-08-03T12%3A59%3A37Z&rscd=attachment%3B+filename%3Dgreen_tripdata_2019-10.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-08-03T11%3A59%3A32Z&ske=2025-08-03T12%3A59%3A37Z&sks=b&skv=2018-11-09&sig=RIkwm79oVueqGkvbspGZzvuClY2EUhHOuuA8eP%2FET%2Bo%3D&jwt=eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc1NDIyMzk0MSwibmJmIjoxNzU0MjIzNjQxLCJw

In [5]:
import pandas as pd

df = pd.read_csv('green_tripdata_2019-10.csv')

/var/folders/2d/272ywp112_39h3qdgj9g8gym0000gn/T/ipykernel_90361/2939510799.py:3: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('green_tripdata_2019-10.csv')


In [55]:
!gunzip ./green_tripdata_2019-10.csv.gz

In [ ]:
for index, row in df.iterrows():
    message = {
        'lpep_pickup_datetime' : row.lpep_pickup_datetime,
        'lpep_dropoff_datetime' : row.lpep_dropoff_datetime,
        'PULocationID' : row.PULocationID,
        'DOLocationID' : row.DOLocationID,
        'passenger_count' : row.passenger_count,
        'trip_distance' : row.trip_distance,
        'tip_amount' : row.tip_amount
    }

    producer.send('green_2019_10', message)
    print(f'sent message for df index no. {index}')

sent message for df index no. 0


In [6]:
len(df)

476386

In [3]:
from kafka.admin import KafkaAdminClient

admin_client = KafkaAdminClient(bootstrap_servers=[server])
admin_client.list_topics()

['green-trips', 'testtopic']

In [57]:
admin_client.delete_topics(['green-trips'])

DeleteTopicsResponse_v3(throttle_time_ms=0, topic_error_codes=[(topic='green-trips', error_code=0)])